# Option Price Dataset Construction

This code creates a daily dataset with four columns:

`date`, `spx`, `r_3m`, `option_price`.

The option price is computed as the mid-price between bid and ask:

$
\text{option\_price}=\frac{\text{best\_bid}+\text{best\_offer}}{2}
$

For each date, the code selects one SPX call option using the target maturity in calendar days and the target moneyness:

$
\text{moneyness}=\frac{S}{K}
$

The selected option is the one closest to the target maturity and target moneyness, giving priority to maturities from the target downward, for example:

$$
30,\ 29,\ 28,\dots
$$

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd


# ================================================================
# 1. File paths
# ================================================================

BASE_FILE = Path("WRDSspx_rf.csv")
OPTIONS_CSV = Path("WRDSoptions.csv")  

OUTPUT_FILE = Path("option_price.csv")


# ================================================================
# 2. Selection settings
# ================================================================

# Moneyness convention:
# moneyness = S / K
#
# Suggested values:
# - 1.00 = ATM call
# - 0.98 = slightly OTM call
# - 0.95 = moderately OTM call
# - 0.90 = deeper OTM call

TARGET_MONEYNESS = 1.00

# Target maturity in calendar days.
# The code gives priority to this value, then goes down:
# 30, 29, 28, ...
TARGET_DAYS_TO_EXPIRY = 30

# Minimum expiry allowed when going down from target.
MIN_DAYS_TO_EXPIRY = 1

# Calls only.
OPTION_TYPE = "C"

# Liquidity / quality filters.
MIN_MID_PRICE = 0.05
MIN_OPEN_INTEREST = 1
MIN_VOLUME = 1

# Tolerance around target moneyness.
# For OTM call target 0.95, this allows options from 0.95 up to below 1.00.
# For ATM target 1.00, the code automatically searches around 1.00.
MONEYNESS_UPPER_BOUND = 1.00


# ================================================================
# 3. Load base dataset
# ================================================================

base_df = pd.read_csv(
    BASE_FILE,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    low_memory=False
).copy()

base_df.columns = [str(col).strip().lower() for col in base_df.columns]

print("Base columns:")
print(base_df.columns.tolist())

required_base_cols = ["date", "spx", "r_3m"]
missing_base_cols = [col for col in required_base_cols if col not in base_df.columns]

if missing_base_cols:
    raise ValueError(
        f"Missing required columns in base file: {missing_base_cols}. "
        f"Available columns are: {base_df.columns.tolist()}"
    )

base_df["date"] = pd.to_datetime(base_df["date"], errors="coerce")
base_df["spx"] = pd.to_numeric(base_df["spx"], errors="coerce")
base_df["r_3m"] = pd.to_numeric(base_df["r_3m"], errors="coerce")


base_df = (
    base_df[["date", "spx", "r_3m"]]
    .dropna(subset=["date", "spx", "r_3m"])
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

print("Base dataframe:")
print(base_df.shape)
display(base_df.head())


# ================================================================
# 4. Load option chain
# ================================================================

options = pd.read_csv(
    OPTIONS_CSV,
    encoding="latin1",
    low_memory=False
).copy()

options.columns = [str(col).strip().lower() for col in options.columns]

required_option_cols = [
    "date",
    "exdate",
    "cp_flag",
    "strike_price",
    "best_bid",
    "best_offer",
]

missing_option_cols = [col for col in required_option_cols if col not in options.columns]

if missing_option_cols:
    raise ValueError(f"Missing required columns in options CSV: {missing_option_cols}")

options["date"] = pd.to_datetime(options["date"], errors="coerce")
options["exdate"] = pd.to_datetime(options["exdate"], errors="coerce")
options["cp_flag"] = options["cp_flag"].astype(str).str.upper().str.strip()

numeric_cols = [
    "strike_price",
    "best_bid",
    "best_offer",
    "volume",
    "open_interest",
    "impl_volatility",
    "delta",
    "vega",
    "forward_price",
]

for col in numeric_cols:
    if col in options.columns:
        options[col] = pd.to_numeric(options[col], errors="coerce")


# ================================================================
# 5. Construct option variables
# ================================================================

# OptionMetrics convention: strike_price is scaled by 1000.
options["strike"] = options["strike_price"] / 1000

# Mid price between bid and ask.
options["mid_price"] = (options["best_bid"] + options["best_offer"]) / 2

# Calendar days to expiry.
options["days_to_expiry"] = (options["exdate"] - options["date"]).dt.days


# ================================================================
# 6. Basic filtering
# ================================================================

options = options[
    options["date"].notna()
    & options["exdate"].notna()
    & options["cp_flag"].eq(OPTION_TYPE)
    & (options["strike"] > 0)
    & (options["best_bid"] >= 0)
    & (options["best_offer"] > options["best_bid"])
    & (options["mid_price"] >= MIN_MID_PRICE)
    & (options["days_to_expiry"] <= TARGET_DAYS_TO_EXPIRY)
    & (options["days_to_expiry"] >= MIN_DAYS_TO_EXPIRY)
].copy()

if "open_interest" in options.columns:
    options = options[options["open_interest"] >= MIN_OPEN_INTEREST].copy()

print("Filtered call options:")
print(options.shape)


# ================================================================
# 7. Merge SPX and compute moneyness
# ================================================================

options = options.merge(
    base_df[["date", "spx"]],
    on="date",
    how="inner",
)

options["moneyness"] = options["spx"] / options["strike"]


# ================================================================
# 8. Moneyness filter
# ================================================================

if TARGET_MONEYNESS < 1.0:
    # OTM call case:
    # Select calls with S/K from target up to below 1.
    candidate_options = options[
        (options["moneyness"] >= TARGET_MONEYNESS)
        & (options["moneyness"] < MONEYNESS_UPPER_BOUND)
    ].copy()

else:
    # ATM case:
    # Select options around S/K = 1.
    # You can widen or tighten this band.
    ATM_LOWER_BOUND = 0.98
    ATM_UPPER_BOUND = 1.02

    candidate_options = options[
        (options["moneyness"] >= ATM_LOWER_BOUND)
        & (options["moneyness"] <= ATM_UPPER_BOUND)
    ].copy()


candidate_options["days_distance"] = (
    TARGET_DAYS_TO_EXPIRY - candidate_options["days_to_expiry"]
)

candidate_options["moneyness_distance"] = (
    candidate_options["moneyness"] - TARGET_MONEYNESS
).abs()

candidate_options["bid_ask_spread"] = (
    candidate_options["best_offer"] - candidate_options["best_bid"]
)

print("Candidate options after moneyness filter:")
print(candidate_options.shape)


# ================================================================
# 9. Select one option per date
# ================================================================

# Priority:
# 1. Expiry closest to target from below: 30, 29, 28, ...
# 2. Moneyness closest to target.
# 3. Smaller bid-ask spread.
# 4. Higher open interest.
# 5. Higher volume.

sort_cols = [
    "date",
    "days_distance",
    "moneyness_distance",
    "bid_ask_spread",
]

ascending = [True, True, True, True]

if "open_interest" in candidate_options.columns:
    sort_cols.append("open_interest")
    ascending.append(False)

if "volume" in candidate_options.columns:
    sort_cols.append("volume")
    ascending.append(False)

selected_options = (
    candidate_options
    .sort_values(sort_cols, ascending=ascending)
    .groupby("date", as_index=False)
    .first()
)

print("Selected options:")
print(selected_options.shape)

display(
    selected_options[
        [
            "date",
            "exdate",
            "spx",
            "strike",
            "moneyness",
            "days_to_expiry",
            "best_bid",
            "best_offer",
            "mid_price",
        ]
    ].head()
)


# ================================================================
# 10. Build final dataset
# ================================================================

selected_prices = selected_options[
    ["date", "mid_price", "strike", "exdate"]
].copy()

selected_prices = selected_prices.rename(
    columns={
        "mid_price": "option_price",
        "exdate": "expiry_date",
    }
)

final_df = base_df.merge(
    selected_prices,
    on="date",
    how="left",
)

final_df = final_df[
    ["date", "spx", "r_3m", "strike", "expiry_date", "option_price"]
].copy()

# Remove time component from dates
final_df["date"] = pd.to_datetime(final_df["date"]).dt.strftime("%Y-%m-%d")
final_df["expiry_date"] = pd.to_datetime(
    final_df["expiry_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# Round numeric columns
final_df["spx"] = final_df["spx"].round(6)
final_df["r_3m"] = final_df["r_3m"].round(6)
final_df["strike"] = final_df["strike"].round(2)
final_df["option_price"] = final_df["option_price"].round(2)

print("Final dataframe:")
print(final_df.shape)

print("Missing option prices:")
print(final_df["option_price"].isna().sum())

display(final_df.head())
display(final_df.tail())


# ================================================================
# 11. Save output
# ================================================================

final_df.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=";",
    decimal=","
)

print(f"Saved file: {OUTPUT_FILE.resolve()}")

Base columns:
['date', 'spx', 'r_3m']
Base dataframe:
(2498, 3)


,date,spx,r_3m
0,2015-01-05,2020.577884,0.0003
1,2015-01-06,2002.613588,0.0003
2,2015-01-07,2025.901051,0.0003
3,2015-01-08,2062.143555,0.0003
4,2015-01-09,2044.809900,0.0002


Filtered call options:
(3985634, 20)
Candidate options after moneyness filter:
(936975, 25)
Selected options:
(2498, 25)


,date,exdate,spx,strike,moneyness,days_to_expiry,best_bid,best_offer,mid_price
0,2015-01-05,2015-01-30,2020.577884,2020.0,1.000286,25,35.1,36.3,35.70
1,2015-01-06,2015-01-30,2002.613588,2005.0,0.998810,24,34.3,35.6,34.95
2,2015-01-07,2015-02-06,2025.901051,2025.0,1.000445,30,36.5,37.4,36.95
3,2015-01-08,2015-02-06,2062.143555,2060.0,1.001041,29,32.7,33.3,33.00
4,2015-01-09,2015-02-06,2044.809900,2045.0,0.999907,28,32.0,33.0,32.50


Final dataframe:
(2498, 6)
Missing option prices:
0


,date,spx,r_3m,strike,expiry_date,option_price
0,2015-01-05,2020.577884,0.0003,2020.0,2015-01-30,35.70
1,2015-01-06,2002.613588,0.0003,2005.0,2015-01-30,34.95
2,2015-01-07,2025.901051,0.0003,2025.0,2015-02-06,36.95
3,2015-01-08,2062.143555,0.0003,2060.0,2015-02-06,33.00
4,2015-01-09,2044.809900,0.0002,2045.0,2015-02-06,32.50


,date,spx,r_3m,strike,expiry_date,option_price
2493,2024-12-24,6040.035577,0.0428,6040.0,2025-01-23,80.40
2494,2024-12-26,6037.590858,0.0424,6040.0,2025-01-24,84.80
2495,2024-12-27,5970.837644,0.0419,5970.0,2025-01-24,91.90
2496,2024-12-30,5906.935600,0.0423,5900.0,2025-01-29,110.20
2497,2024-12-31,5881.627648,0.0423,5890.0,2025-01-30,96.95


Saved file: /Users/paridelauretti/Desktop/thesis/code/feature_engineering/option_price.csv
